In [1]:
import pandas as pd
import numpy as np

# ── 1. LOAD — try common separators ─────────────────────────────────────────
try:
    df = pd.read_csv("data/gene properties/5_OmicsFusionFilteredSupplementary.csv")
    if df.shape[1] == 1:
        raise ValueError
except:
    df = pd.read_csv("data/gene properties/5_OmicsFusionFilteredSupplementary.csv", sep="\t")

print("=== SHAPE ===")
print(df.shape)

# ── 2. COLUMNS & DTYPES ──────────────────────────────────────────────────────
print("\n=== COLUMNS & DTYPES ===")
print(df.dtypes.to_string())

# ── 3. FIRST FEW ROWS ────────────────────────────────────────────────────────
print("\n=== HEAD (5) ===")
print(df.head(5).to_string())

# ── 4. NULLS ─────────────────────────────────────────────────────────────────
print("\n=== NULL COUNTS ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0])
print(f"Total null cells: {df.isnull().sum().sum():,}")



=== SHAPE ===
(184237, 30)

=== COLUMNS & DTYPES ===
Unnamed: 0                      int64
SequencingID                   object
ModelID                        object
IsDefaultEntryForModel         object
ModelConditionID               object
IsDefaultEntryForMC            object
CanonicalFusionName            object
gene1(ENS ID)                  object
gene2(ENS ID)                  object
TotalReadsInSample              int64
TotalReadsSupportingFusion      int64
FFPM                          float64
confidence                     object
split_reads1                    int64
split_reads2                    int64
discordant_mates                int64
strand1(gene/fusion)           object
strand2(gene/fusion)           object
reading_frame                  object
breakpoint1                    object
breakpoint2                    object
site1                          object
site2                          object
type                           object
coverage1                       int

In [5]:
# ── 5. CELL LINE ID FORMAT ───────────────────────────────────────────────────
print("\n=== CELL LINE ID FORMAT ===")
for col in df.columns[:5]:
    sample = df[col].dropna().astype(str).head(5).tolist()
    has_ach  = any(v.startswith('ACH-') for v in sample)
    has_cvcl = any('CVCL' in v for v in sample)
    has_pr   = any(v.startswith('PR-') for v in sample)
    print(f"  {col}: {sample[:3]}  ACH={has_ach} CVCL={has_cvcl} PR={has_pr}")




=== CELL LINE ID FORMAT ===
  Unnamed: 0: ['0', '1', '2']  ACH=False CVCL=False PR=False
  SequencingID: ['CDS-010xbm', 'CDS-010xbm', 'CDS-010xbm']  ACH=False CVCL=False PR=False
  ModelID: ['ACH-001113', 'ACH-001113', 'ACH-001113']  ACH=True CVCL=False PR=False
  IsDefaultEntryForModel: ['Yes', 'Yes', 'Yes']  ACH=False CVCL=False PR=False
  ModelConditionID: ['MC-001113-k2lR', 'MC-001113-k2lR', 'MC-001113-k2lR']  ACH=False CVCL=False PR=False


In [2]:
# ── 6. FUSION FORMAT ─────────────────────────────────────────────────────────
# Fusions are typically stored as GENE1--GENE2 or GENE1_GENE2
print("\n=== FUSION FORMAT SAMPLE ===")
for col in df.columns:
    vals = df[col].dropna().astype(str)
    fusion_like = vals[vals.str.contains('--', na=False)]
    if len(fusion_like) > 0:
        print(f"  Column '{col}' contains '--' notation: {fusion_like.head(3).tolist()}")
    fusion_like2 = vals[vals.str.contains('_', na=False)]
    if len(fusion_like2) > 0:
        print(f"  Column '{col}' contains '_' notation: {fusion_like2.head(3).tolist()}")




=== FUSION FORMAT SAMPLE ===
  Column 'CanonicalFusionName' contains '--' notation: ['DLG1--SERPINI1', 'ADAM17--ITGB1BP1', 'ADAM17--ITGB1BP1']
  Column 'CanonicalFusionName' contains '_' notation: ['HNRNPA1P48--Y_RNA', 'CTDNEP1--Y_RNA', 'RP1-4G17.5--Y_RNA']
  Column 'gene1(ENS ID)' contains '_' notation: ['Y_RNA (ENSG00000206995.1)', 'Y_RNA (.)', 'Y_RNA (.)']
  Column 'gene2(ENS ID)' contains '_' notation: ['Y_RNA (.)', 'Y_RNA (.)', 'Y_RNA (.)']
  Column 'breakpoint1' contains '_' notation: ['NC_001802.1:180', 'NC_001802.1:7625', 'NC_001802.1:4348']
  Column 'breakpoint2' contains '_' notation: ['NC_007605.1:40193', 'NC_001802.1:178', 'NC_001436.1:8397']
  Column 'type' contains '_' notation: ['duplication/non-canonical_splicing', 'duplication/non-canonical_splicing', 'duplication/non-canonical_splicing']


In [3]:
# ── 7. UNIQUE VALUES IN KEY COLUMNS ─────────────────────────────────────────
print("\n=== UNIQUE VALUE COUNTS PER COLUMN ===")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} unique")

# ── 8. FORMAT GUESS — long vs wide vs event list ────────────────────────────
print("\n=== FORMAT GUESS ===")
print(f"Rows: {df.shape[0]:,}  Cols: {df.shape[1]}")
if df.shape[0] > df.shape[1]:
    print("→ Likely LONG format (one row per fusion event)")
else:
    print("→ Likely WIDE format")




=== UNIQUE VALUE COUNTS PER COLUMN ===
  Unnamed: 0: 184237 unique
  SequencingID: 1754 unique
  ModelID: 1699 unique
  IsDefaultEntryForModel: 2 unique
  ModelConditionID: 1700 unique
  IsDefaultEntryForMC: 2 unique
  CanonicalFusionName: 72848 unique
  gene1(ENS ID): 21117 unique
  gene2(ENS ID): 30433 unique
  TotalReadsInSample: 1734 unique
  TotalReadsSupportingFusion: 850 unique
  FFPM: 52993 unique
  confidence: 3 unique
  split_reads1: 319 unique
  split_reads2: 324 unique
  discordant_mates: 301 unique
  strand1(gene/fusion): 9 unique
  strand2(gene/fusion): 9 unique
  reading_frame: 4 unique
  breakpoint1: 71350 unique
  breakpoint2: 89669 unique
  site1: 12 unique
  site2: 12 unique
  type: 17 unique
  coverage1: 10300 unique
  coverage2: 10061 unique
  tags: 1 unique
  retained_protein_domains: 1 unique
  direction1: 2 unique
  direction2: 2 unique

=== FORMAT GUESS ===
Rows: 184,237  Cols: 30
→ Likely LONG format (one row per fusion event)


In [4]:
# ── 9. GENE COVERAGE ─────────────────────────────────────────────────────────
# Check if canonical cancer fusion genes are present
print("\n=== CANONICAL FUSION SPOT CHECK ===")
df_str = df.astype(str)
for fusion in ['BCR', 'ABL1', 'EML4', 'ALK', 'TMPRSS2', 'ERG']:
    found = df_str.apply(lambda col: col.str.contains(fusion, na=False)).any().any()
    print(f"  {fusion}: {'FOUND' if found else 'not found'}")

# ── 10. CELL LINE COVERAGE ───────────────────────────────────────────────────
print("\n=== CELL LINE COVERAGE ===")
# Find the column most likely to be the cell line identifier
for col in df.columns:
    sample = df[col].dropna().astype(str).head(10)
    if sample.str.startswith('ACH-').any():
        print(f"  Cell line column: '{col}'")
        print(f"  Unique cell lines: {df[col].nunique()}")
        break


=== CANONICAL FUSION SPOT CHECK ===
  BCR: FOUND
  ABL1: FOUND
  EML4: FOUND
  ALK: FOUND
  TMPRSS2: FOUND
  ERG: FOUND

=== CELL LINE COVERAGE ===
  Cell line column: 'ModelID'
  Unique cell lines: 1699


In [7]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("data/gene properties/5_OmicsFusionFilteredSupplementary.csv")

# ── 1. DEFAULT ENTRY FILTER IMPACT ──────────────────────────────────────────
print("=== IsDefaultEntryForModel ===")
print(df['IsDefaultEntryForModel'].value_counts())
df_default = df[df['IsDefaultEntryForModel'] == 'Yes']
print(f"Rows after filtering to default: {len(df_default):,}")
print(f"Unique cell lines after filter:  {df_default['ModelID'].nunique()}")

# ── 2. CONFIDENCE DISTRIBUTION ───────────────────────────────────────────────
print("\n=== CONFIDENCE LEVELS ===")
print(df_default['confidence'].value_counts())
print("\nBy confidence — what % of fusions are high confidence?")
conf_pct = df_default['confidence'].value_counts(normalize=True) * 100
print(conf_pct.round(1))

# ── 3. FFPM DISTRIBUTION ─────────────────────────────────────────────────────
print("\n=== FFPM DISTRIBUTION ===")
print(df_default['FFPM'].describe())
print(f"FFPM = 0:        {(df_default['FFPM'] == 0).sum()}")
print(f"FFPM < 0.1:      {(df_default['FFPM'] < 0.1).sum():,}")
print(f"FFPM >= 1.0:     {(df_default['FFPM'] >= 1.0).sum():,}")
print(f"FFPM >= 10.0:    {(df_default['FFPM'] >= 10.0).sum():,}")

# ── 4. FUSIONS PER CELL LINE ─────────────────────────────────────────────────
print("\n=== FUSIONS PER CELL LINE ===")
fusions_per_line = df_default.groupby('ModelID').size()
print(fusions_per_line.describe())
print(f"Cell lines with 0 fusions (not in file): n/a — all have at least 1")
print(f"Cell lines with >100 fusions: {(fusions_per_line > 100).sum()}")
print(f"Top 5 fusion-heavy lines:\n{fusions_per_line.nlargest(5)}")

# ── 5. READING FRAME DISTRIBUTION ───────────────────────────────────────────
print("\n=== READING FRAME ===")
print(df_default['reading_frame'].value_counts())
print("In-frame fusions are most biologically significant")

# ── 6. FUSION TYPE DISTRIBUTION ─────────────────────────────────────────────
print("\n=== FUSION TYPES ===")
print(df_default['type'].value_counts())

# ── 7. ENSG PARSING FROM GENE COLUMNS ───────────────────────────────────────
print("\n=== ENSG PARSING ===")
def extract_ensg(val):
    m = re.search(r'(ENSG\d+)', str(val))
    return m.group(1) if m else None

sample = df_default['gene1(ENS ID)'].head(10)
for v in sample:
    print(f"  Raw: {v!r:45s}  →  ENSG: {extract_ensg(v)}")

dot_count1 = (df_default['gene1(ENS ID)'].str.contains(r'\(\.\)', na=False)).sum()
dot_count2 = (df_default['gene2(ENS ID)'].str.contains(r'\(\.\)', na=False)).sum()
print(f"\nRows where gene1 ENSG is '.': {dot_count1:,}")
print(f"Rows where gene2 ENSG is '.': {dot_count2:,}")
print("These are intergenic/unknown partners — need handling at integration")

# ── 8. CANONICAL FUSION SPOT CHECK (high confidence only) ────────────────────
print("\n=== CANONICAL FUSIONS (high confidence, default entry) ===")
df_high = df_default[df_default['confidence'] == 'high']
for pair in [('BCR', 'ABL1'), ('EML4', 'ALK'), ('TMPRSS2', 'ERG')]:
    g1, g2 = pair
    hits = df_high[df_high['CanonicalFusionName'].str.contains(f'{g1}--{g2}', na=False)]
    print(f"  {g1}--{g2}: {len(hits)} events across "
          f"{hits['ModelID'].nunique()} cell lines")

# ── 9. TAGS AND RETAINED DOMAINS — confirm they're empty ────────────────────
print("\n=== TAGS & RETAINED_PROTEIN_DOMAINS ===")
print(f"tags unique values:                    {df['tags'].unique()}")
print(f"retained_protein_domains unique values: {df['retained_protein_domains'].unique()}")
print("Both are single-value columns — drop at integration")

# ── 10. ROLE IN CELLLINEFINDER ───────────────────────────────────────────────
print("\n=== ROLE IN CELLLINEFINDER ===")
print("Fusions are contextual flags, not expression evidence.")
print("Use case: user queries GENE_A — check if any high-confidence fusion")
print("involving GENE_A exists in candidate cell lines.")
print("Flag in AI evidence card as trade-off or confound.")
print("Example: HEK293 expresses EGFR but also carries EGFRvIII fusion — flag.")

=== IsDefaultEntryForModel ===
IsDefaultEntryForModel
Yes    178041
No       6196
Name: count, dtype: int64
Rows after filtering to default: 178,041
Unique cell lines after filter:  1699

=== CONFIDENCE LEVELS ===
confidence
low       73687
high      53583
medium    50771
Name: count, dtype: int64

By confidence — what % of fusions are high confidence?
confidence
low       41.4
high      30.1
medium    28.5
Name: proportion, dtype: float64

=== FFPM DISTRIBUTION ===
count    178041.000000
mean          0.372834
std           1.122009
min           0.000000
25%           0.040669
50%           0.076331
75%           0.215189
max          27.339642
Name: FFPM, dtype: float64
FFPM = 0:        6172
FFPM < 0.1:      103,609
FFPM >= 1.0:     13,854
FFPM >= 10.0:    522

=== FUSIONS PER CELL LINE ===
count    1699.000000
mean      104.791642
std        72.586345
min         5.000000
25%        57.000000
50%        88.000000
75%       134.000000
max       797.000000
dtype: float64
Cell lines w